### Imports
Import Python modules for executing the notebook.
- `pandas` is used for building and handling dataframes.
- `scrapbook` is used for recording data for the Notebook Execution Service.
- `WorkItemClient` is the NI SystemLink Python client for managing work items.

In [ ]:
import pandas as pd
import scrapbook as sb
from nisystemlink.clients.core import HttpConfiguration
from nisystemlink.clients.work_item import WorkItemClient
from nisystemlink.clients.work_item.models import (
    QueryWorkItemsRequest,
    UpdateWorkItemRequest,
    UpdateWorkItemsRequest,
)

### Parameters
- `work_item_ids`: IDs of the work items to be closed (passed as a list by SystemLink).
- `workspace`: The workspace name or ID containing the work items. Leave empty for the default workspace.
- `closed_state`: The target state name to set on each work item (default: `CLOSED`).

In [ ]:
# Add the work item IDs as a list of strings here
work_item_ids = []

workspace = ""
closed_state = "CLOSED"

### Initialize Client
Create the WorkItemClient using the default SystemLink connection.

In [ ]:
# Server configuration is not required when run through Jupyter on SystemLink
server_configuration: HttpConfiguration | None = None

# To connect to a remote SystemLink server, uncomment and fill in:
# server_configuration = HttpConfiguration(
#     server_uri="https://yourserver.yourcompany.com",
#     api_key="",
# )

client = WorkItemClient(configuration=server_configuration)

### Query Work Items to Close
Parse the comma-separated list of work item IDs and retrieve each work item to confirm it exists and check its current state.

In [ ]:
if not work_item_ids:
    raise ValueError("No work item IDs provided. Select one or more work items before running.")

if len(work_item_ids) > 1000:
    raise ValueError("Update limit exceeded: Up to 1000 work items can be updated at a time.")

# Retrieve each work item and record its current state
work_items_before = []
for wid in work_item_ids:
    item = client.get_work_item(work_item_id=wid)
    work_items_before.append({"id": item.id, "name": item.name, "state_before": str(item.state)})

df_before = pd.DataFrame(work_items_before)
print(f"Found {len(df_before)} work item(s) to close:")
df_before

### Update Work Item Status to Closed
Send a batch update request to change the state of each work item to the closed state.

In [ ]:
update_requests = [
    UpdateWorkItemRequest(id=wid, state=closed_state)
    for wid in work_item_ids
]

update_response = client.update_work_items(
    update_work_items=UpdateWorkItemsRequest(work_items=update_requests)
)

updated_count = len(update_response.updated_work_items) if update_response.updated_work_items else 0
failed_count = len(update_response.failed_work_items) if update_response.failed_work_items else 0

print(f"Updated: {updated_count}, Failed: {failed_count}")

if update_response.failed_work_items:
    for failure in update_response.failed_work_items:
        print(f"  Failed ID: {failure.id}, Error: {failure.error}")

### Verify Status Changes
Re-query the updated work items to confirm their state has been changed, and display a summary.

In [ ]:
work_items_after = []
for wid in work_item_ids:
    item = client.get_work_item(work_item_id=wid)
    work_items_after.append({"id": item.id, "name": item.name, "state_after": str(item.state)})

df_after = pd.DataFrame(work_items_after)
df_summary = df_before.merge(df_after, on=["id", "name"])
print(f"Summary of {len(df_summary)} work item(s):")
df_summary

### Actions and output
Print the script output and send results to the Execution Results page via Scrapbook. The output includes:
1. The total number of work items processed.
2. A list of work item IDs that were successfully closed.
3. A list of work item IDs that were not closed.

In [ ]:
updated_ids = [row["id"] for _, row in df_summary.iterrows() if row["state_after"] == closed_state]
failed_ids = [row["id"] for _, row in df_summary.iterrows() if row["state_after"] != closed_state]

# Print output
print("Total work items:", len(work_item_ids))
print("Work items closed:", ", ".join(updated_ids) if updated_ids else "--")
print("Work items not closed:", ", ".join(failed_ids) if failed_ids else "--")

# Executions page output
sb.glue("Total work items:", len(work_item_ids))
sb.glue("Work items closed:", ", ".join(updated_ids) if updated_ids else "--")
sb.glue("Work items not closed:", ", ".join(failed_ids) if failed_ids else "--")